# Import Library Load Dataset

In [59]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer


In [43]:

df = pd.read_csv('D:/vishal7029/auto-mpg.csv')
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


# Data Cleaning and Fill Null values

In [44]:


df.replace('?',np.nan,inplace=True)
df['horsepower'] = pd.to_numeric(df['horsepower'])
df.dropna()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130.0,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,1,ford torino
...,...,...,...,...,...,...,...,...,...
393,27.0,4,140.0,86.0,2790,15.6,82,1,ford mustang gl
394,44.0,4,97.0,52.0,2130,24.6,82,2,vw pickup
395,32.0,4,135.0,84.0,2295,11.6,82,1,dodge rampage
396,28.0,4,120.0,79.0,2625,18.6,82,1,ford ranger


In [45]:
if 'car name' in df.columns:
  df = df.drop(columns=['car name'])

# Train & Test Model

In [46]:
X = df.drop(columns=['mpg'])
y = df['mpg']

In [47]:
X.head()

,cylinders,displacement,horsepower,weight,acceleration,model year,origin
0,8,307.0,130.0,3504,12.0,70,1
1,8,350.0,165.0,3693,11.5,70,1
2,8,318.0,150.0,3436,11.0,70,1
3,8,304.0,150.0,3433,12.0,70,1
4,8,302.0,140.0,3449,10.5,70,1


In [48]:
y.head()

0    18.0
1    15.0
2    18.0
3    16.0
4    17.0
Name: mpg, dtype: float64

In [49]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [50]:
X_train.head()

,cylinders,displacement,horsepower,weight,acceleration,model year,origin
3,8,304.0,150.0,3433,12.0,70,1
18,4,97.0,88.0,2130,14.5,70,3
376,4,91.0,68.0,2025,18.2,82,3
248,4,91.0,60.0,1800,16.4,78,3
177,4,115.0,95.0,2694,15.0,75,2


In [51]:
X_test.head()

,cylinders,displacement,horsepower,weight,acceleration,model year,origin
198,4,91.0,53.0,1795,17.4,76,3
396,4,120.0,79.0,2625,18.6,82,1
33,6,232.0,100.0,2634,13.0,71,1
208,8,318.0,150.0,3940,13.2,76,1
93,8,318.0,150.0,4237,14.5,73,1


In [52]:
y_train.head()

3      16.0
18     27.0
376    37.0
248    36.1
177    23.0
Name: mpg, dtype: float64

In [53]:
y_test.head()

198    33.0
396    28.0
33     19.0
208    13.0
93     14.0
Name: mpg, dtype: float64

# Features

In [54]:
#remaining_features → features not yet selected
#selected_features → features already selected
#best_score → best R² score achieved so far
#We initialize best_score = -∞ so that the first feature always improves the score.

remaining_features = list(X.columns)
selected_features = []
best_score = -np.inf

# Forward Feature Selection Process:

In [58]:
while remaining_features:
    scores = []

   #We temporarily add one feature at a time to the selected set. 
    for feature in remaining_features:
        features_to_test = selected_features + [feature]
        imputer = SimpleImputer(strategy='mean')

        X_train_imputed = imputer.fit_transform(X_train[features_to_test])
        X_test_imputed = imputer.transform(X_test[features_to_test])

        
        model = LinearRegression()
        model.fit(X_train_imputed, y_train)
        y_pred = model.predict(X_test_imputed)
        
        #Train regression model using only the selected + candidate feature.
        #This is the wrapper mechanism (model-based evaluation).
       
       
        
        score = r2_score(y_test, y_pred)
        
        scores.append((score, feature))
    
    #Sort features by R² score.
    #Select feature giving highest improvement.
    scores.sort(reverse=True)
    current_best_score, best_feature = scores[0]


#If adding the feature improves R²:
#Update best_score
#Add feature permanently
#Remove from remaining list
#Else:
#Stop algorithm (no further improvement)

    
    if current_best_score > best_score:
        best_score = current_best_score
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
        
        print(f"Added: {best_feature}, R2 Score: {best_score:.4f}")
    else:
        break

print("\nFinal Selected Features:")
print(selected_features)

Added: weight, R2 Score: 0.7230
Added: model year, R2 Score: 0.8244
Added: origin, R2 Score: 0.8409
Added: acceleration, R2 Score: 0.8421
Added: displacement, R2 Score: 0.8455
Added: cylinders, R2 Score: 0.8473
Added: horsepower, R2 Score: 0.8476

Final Selected Features:
['weight', 'model year', 'origin', 'acceleration', 'displacement', 'cylinders', 'horsepower']
